In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, hashlib

PATH = "/content/drive/MyDrive/nfton/data/NF-ToN-IoT-v3.csv"

# 1) integrity check against the manifest
h = hashlib.sha1()
with open(PATH, "rb") as f:
    for block in iter(lambda: f.read(1024*1024*64), b""):
        h.update(block)
print("SHA-1:", h.hexdigest())
print("Expected: 24854ca3072ab7c2ade9ebfb202345a666a94cfe")

# 2) look at the columns
head = pd.read_csv(PATH, nrows=5)
print("\nCOLUMNS:", list(head.columns))
print(head.T)

# 3) stream the file once: count labels and find the time range
label_cols = [c for c in head.columns if "label" in c.lower() or "attack" in c.lower()]
time_cols  = [c for c in head.columns if "time" in c.lower() or "start" in c.lower()]
print("\nLabel-like columns:", label_cols, "| Time-like columns:", time_cols)

counts, tmin, tmax, total = {}, None, None, 0
for chunk in pd.read_csv(PATH, chunksize=1_000_000,
                         usecols=label_cols + time_cols[:1]):
    total += len(chunk)
    for c in label_cols:
        counts.setdefault(c, pd.Series(dtype="int64"))
        counts[c] = counts[c].add(chunk[c].value_counts(), fill_value=0)
    if time_cols:
        t = chunk[time_cols[0]]
        tmin = t.min() if tmin is None else min(tmin, t.min())
        tmax = t.max() if tmax is None else max(tmax, t.max())

print("\nTotal rows:", total)
for c, s in counts.items():
    print("\n", c); print(s.astype(int).sort_values(ascending=False))
print("\nTime range:", tmin, "to", tmax)

Mounted at /content/drive
SHA-1: 24854ca3072ab7c2ade9ebfb202345a666a94cfe
Expected: 24854ca3072ab7c2ade9ebfb202345a666a94cfe

COLUMNS: ['FLOW_START_MILLISECONDS', 'FLOW_END_MILLISECONDS', 'IPV4_SRC_ADDR', 'L4_SRC_PORT', 'IPV4_DST_ADDR', 'L4_DST_PORT', 'PROTOCOL', 'L7_PROTO', 'IN_BYTES', 'IN_PKTS', 'OUT_BYTES', 'OUT_PKTS', 'TCP_FLAGS', 'CLIENT_TCP_FLAGS', 'SERVER_TCP_FLAGS', 'FLOW_DURATION_MILLISECONDS', 'DURATION_IN', 'DURATION_OUT', 'MIN_TTL', 'MAX_TTL', 'LONGEST_FLOW_PKT', 'SHORTEST_FLOW_PKT', 'MIN_IP_PKT_LEN', 'MAX_IP_PKT_LEN', 'SRC_TO_DST_SECOND_BYTES', 'DST_TO_SRC_SECOND_BYTES', 'RETRANSMITTED_IN_BYTES', 'RETRANSMITTED_IN_PKTS', 'RETRANSMITTED_OUT_BYTES', 'RETRANSMITTED_OUT_PKTS', 'SRC_TO_DST_AVG_THROUGHPUT', 'DST_TO_SRC_AVG_THROUGHPUT', 'NUM_PKTS_UP_TO_128_BYTES', 'NUM_PKTS_128_TO_256_BYTES', 'NUM_PKTS_256_TO_512_BYTES', 'NUM_PKTS_512_TO_1024_BYTES', 'NUM_PKTS_1024_TO_1514_BYTES', 'TCP_WIN_MAX_IN', 'TCP_WIN_MAX_OUT', 'ICMP_TYPE', 'ICMP_IPV4_TYPE', 'DNS_QUERY_ID', 'DNS_QUERY_TYPE'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd

PATH = "/content/drive/MyDrive/nfton/data/NF-ToN-IoT-v3.csv"

parts = []
for chunk in pd.read_csv(PATH, chunksize=2_000_000,
                         usecols=["FLOW_START_MILLISECONDS", "Attack"]):
    day = pd.to_datetime(chunk["FLOW_START_MILLISECONDS"], unit="ms", utc=True).dt.strftime("%Y-%m-%d")
    parts.append(pd.crosstab(day, chunk["Attack"]))

table = pd.concat(parts).groupby(level=0).sum().fillna(0).astype(int)
print(table.to_string())
table.to_csv("/content/drive/MyDrive/nfton/day_by_class.csv")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Attack                    Benign  scanning     dos     ddos  injection  password      xss  Backdoor  mitm  ransomware
FLOW_START_MILLISECONDS                                                                                              
2019-04-23               5580827    262207       0        0          0         0        0         0     0           0
2019-04-24               3108051   1096770  133307        0          0         0        0         0     0           0
2019-04-25               3434087         0   70149  2487061     381777         0        0         0     0           0
2019-04-26               2140689         0       0  1654195          0   1032571        0         0     0           0
2019-04-27                354308         0       0        0          0    562206  2834435         0     0           0
2019-04-28                215980         0   

In [ ]:
import pandas as pd, numpy as np
from google.colab import drive
drive.mount('/content/drive')

PATH = "/content/drive/MyDrive/nfton/data/NF-ToN-IoT-v3.csv"
OUT  = "/content/drive/MyDrive/nfton/working_set.parquet"
SEED = 2026   # fixed sampling seed; report it in the paper

# (day, class, rows in full file, rows to keep)
PLAN = [
 ("2019-04-23","Benign",5580827,600000),
 ("2019-04-23","scanning",262207,100000),
 ("2019-04-24","Benign",3108051,300000),
 ("2019-04-24","dos",133307,133307),
 ("2019-04-25","Benign",3434087,300000),
 ("2019-04-25","dos",70149,70149),
 ("2019-04-25","ddos",2487061,100000),
 ("2019-04-26","Benign",2140689,300000),
 ("2019-04-26","ddos",1654195,100000),
 ("2019-04-27","Benign",354308,300000),
 ("2019-04-27","xss",2834435,200000),
 ("2019-04-28","Benign",215980,215980),
 ("2019-04-29","Benign",1958272,300000),
]
KEEP = {(d, c): min(1.0, k / t) for d, c, t, k in PLAN}

DROP = ["FLOW_END_MILLISECONDS", "IPV4_SRC_ADDR", "IPV4_DST_ADDR",
        "L4_SRC_PORT", "L4_DST_PORT", "DNS_QUERY_ID"]

rng = np.random.default_rng(SEED)
kept = []
for chunk in pd.read_csv(PATH, chunksize=1_000_000):
    chunk["day"] = pd.to_datetime(chunk["FLOW_START_MILLISECONDS"],
                                  unit="ms", utc=True).dt.strftime("%Y-%m-%d")
    p = np.array([KEEP.get((d, a), 0.0) for d, a in zip(chunk["day"], chunk["Attack"])])
    kept.append(chunk[rng.random(len(chunk)) < p].drop(columns=DROP))
    print("kept so far:", sum(len(k) for k in kept))

ws = pd.concat(kept, ignore_index=True)
ws = ws.sort_values("FLOW_START_MILLISECONDS").reset_index(drop=True)
ws.to_parquet(OUT)
print("\nSaved:", OUT, ws.shape)
print(ws.groupby(["day", "Attack"]).size())
print("\nMissing values:", int(ws.isna().sum().sum()),
      "| Infinite values:", int(np.isinf(ws.select_dtypes("number")).sum().sum()))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
kept so far: 176414
kept so far: 284189
kept so far: 392043
kept so far: 500370
kept so far: 609198
kept so far: 710898
kept so far: 776758
kept so far: 857443
kept so far: 951192
kept so far: 1007146
kept so far: 1231494
kept so far: 1296969
kept so far: 1362985
kept so far: 1430741
kept so far: 1498261
kept so far: 1566095
kept so far: 1650697
kept so far: 1758113
kept so far: 1862328
kept so far: 1962143
kept so far: 1998239
kept so far: 2032815
kept so far: 2256809
kept so far: 2373110
kept so far: 2487346
kept so far: 2786530
kept so far: 2939489
kept so far: 3019057

Saved: /content/drive/MyDrive/nfton/working_set.parquet (3019057, 50)
day         Attack  
2019-04-23  Benign      600336
            scanning    100378
2019-04-24  Benign      299974
            dos         133307
2019-04-25  Benign      299238
            ddos         99891
            do

In [ ]:
%%writefile /content/drive/MyDrive/nfton/common.py
import numpy as np, pandas as pd
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score

DEV_DAY = "2019-04-23"
ID_LIKE = ["FLOW_START_MILLISECONDS", "day", "Label", "Attack"]
CAT = ["PROTOCOL", "L7_PROTO"]
ALPHAS = (0.001, 0.01, 0.05)
BLOCK = 5000

def get_num_cols(df):
    return [c for c in df.columns if c not in ID_LIKE + CAT]

def make_split(ws, seed):
    dev_idx = ws.index[(ws["day"] == DEV_DAY) & (ws["Attack"] == "Benign")].to_numpy()
    block = np.arange(len(dev_idx)) // BLOCK
    nb = int(block.max()) + 1
    order = np.random.default_rng(seed).permutation(nb)
    n_tr, n_va = int(0.6 * nb), int(0.1 * nb)
    role = np.empty(nb, dtype=object)
    role[order[:n_tr]] = "train"
    role[order[n_tr:n_tr + n_va]] = "val"
    role[order[n_tr + n_va:]] = "cal"
    r = role[block]
    return {k: dev_idx[r == k] for k in ["train", "val", "cal"]}

def fit_prep(train_df):
    NUM = get_num_cols(train_df)
    X = train_df[NUM].replace([np.inf, -np.inf], np.nan)
    med = X.median()
    X = X.fillna(med).astype("float64")
    log_cols = [c for c in NUM if X[c].min() >= 0]
    Xl = X.copy()
    Xl[log_cols] = np.log1p(Xl[log_cols])
    std = Xl.std()
    const = [c for c in NUM if std[c] == 0]
    keep = [c for c in NUM if std[c] > 0]
    corr = Xl[keep].corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
    dcorr = [c for c in upper.columns if (upper[c] > 0.98).any()]
    keep = [c for c in keep if c not in dcorr]
    scaler = StandardScaler().fit(Xl[keep])
    ohe = OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=0.01,
                        sparse_output=False).fit(train_df[CAT].astype(str))
    return dict(NUM=NUM, median=med, log_cols=log_cols, num_keep=keep,
                dropped_const=const, dropped_corr=dcorr, scaler=scaler, ohe=ohe)

def apply_prep(p, df, chunk=500000):
    outs = []
    for i in range(0, len(df), chunk):
        d = df.iloc[i:i + chunk]
        X = d[p["NUM"]].replace([np.inf, -np.inf], np.nan).fillna(p["median"]).astype("float64")
        X[p["log_cols"]] = np.log1p(X[p["log_cols"]].clip(lower=0))
        Xn = p["scaler"].transform(X[p["num_keep"]])
        Xc = p["ohe"].transform(d[CAT].astype(str))
        outs.append(np.hstack([Xn, Xc]).astype("float32"))
    return np.vstack(outs)

def audit(ws, split, p):
    allidx = np.concatenate([split["train"], split["val"], split["cal"]])
    assert len(set(allidx)) == len(allidx), "partitions overlap"
    dev = ws.loc[allidx]
    assert (dev["Attack"] == "Benign").all(), "non-benign flow in development"
    assert not dev["Attack"].isin(["dos", "ddos"]).any(), "DoS-family record in development"
    assert (dev["day"] == DEV_DAY).all(), "development data from a later day"
    assert p["scaler"].n_samples_seen_ == len(split["train"]), "scaler not fitted on train only"
    for c in ID_LIKE:
        assert c not in p["num_keep"] and c not in CAT, "forbidden column used as feature"
    return "ALL AUDIT ASSERTIONS PASSED"

def conformal_threshold(cal_scores, alpha):
    s = np.sort(cal_scores)
    n = len(s)
    k = int(np.ceil((n + 1) * (1 - alpha)))
    if k > n:
        return np.inf
    return s[k - 1]

def evaluate(scores, ws, split, model, seed):
    out = []
    def add(metric, group, alpha, value, n):
        out.append(dict(model=model, seed=seed, metric=metric, group=group,
                        alpha=alpha, value=float(value), n=int(n)))
    s = scores.to_numpy()
    day, att = ws["day"].to_numpy(), ws["Attack"].to_numpy()
    later_benign = (att == "Benign") & (day >= "2019-04-24")
    atk = {"dos": att == "dos", "ddos": att == "ddos",
           "dos+ddos": np.isin(att, ["dos", "ddos"]), "xss": att == "xss"}
    cal, val = s[split["cal"]], s[split["val"]]
    for a in ALPHAS:
        tau = conformal_threshold(cal, a)
        add("tau", "calibration", a, tau, len(cal))
        add("FPR", "val_benign_2019-04-23", a, (val > tau).mean(), len(val))
        for d in sorted(set(day[later_benign])):
            m = later_benign & (day == d)
            add("FPR", "benign_" + d, a, (s[m] > tau).mean(), m.sum())
        add("FPR", "benign_later_all", a, (s[later_benign] > tau).mean(), later_benign.sum())
        for g, m in atk.items():
            add("recall", g, a, (s[m] > tau).mean(), m.sum())
    for g, m in atk.items():
        y = np.r_[np.zeros(later_benign.sum()), np.ones(m.sum())]
        sc = np.r_[s[later_benign], s[m]]
        add("AUROC", g, np.nan, roc_auc_score(y, sc), len(y))
        add("AUPRC", g, np.nan, average_precision_score(y, sc), len(y))
    return pd.DataFrame(out)

Writing /content/drive/MyDrive/nfton/common.py


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import numpy as np, pandas as pd, json, sys, joblib, sklearn, time
sys.path.append("/content/drive/MyDrive/nfton")
import importlib, common
importlib.reload(common)
from common import *
from sklearn.ensemble import IsolationForest

SEED = 13
BASE = "/content/drive/MyDrive/nfton"
ws = pd.read_parquet(BASE + "/working_set.parquet").reset_index(drop=True)

split = make_split(ws, SEED)
print("partition sizes:", {k: len(v) for k, v in split.items()})

p = fit_prep(ws.loc[split["train"]])
print(audit(ws, split, p))
print("numeric features kept:", len(p["num_keep"]))
print("dropped (constant):", p["dropped_const"])
print("dropped (correlated):", p["dropped_corr"])

X = apply_prep(p, ws)
print("feature matrix:", X.shape)

rng = np.random.default_rng(SEED)
sub = rng.choice(split["train"], size=min(50000, len(split["train"])), replace=False)
t0 = time.time()
iforest = IsolationForest(n_estimators=200, random_state=SEED, n_jobs=-1).fit(X[sub])

scan23 = ws.index[(ws["day"] == DEV_DAY) & (ws["Attack"] == "scanning")].to_numpy()
need = np.setdiff1d(ws.index.to_numpy(), np.r_[split["train"], scan23])
scores = pd.Series(np.nan, index=ws.index)
for i in range(0, len(need), 500000):
    part = need[i:i + 500000]
    scores.loc[part] = -iforest.score_samples(X[part])
print("scoring done in %.1f min" % ((time.time() - t0) / 60))

res = evaluate(scores, ws, split, "IsolationForest", SEED)
res.to_csv(BASE + f"/results_IsolationForest_seed{SEED}.csv", index=False)

print("\nFPR and recall (columns = nominal false-alarm target):")
print(res[res.metric.isin(["FPR", "recall"])]
      .pivot(index=["metric", "group"], columns="alpha", values="value").round(4).to_string())
print("\nRanking metrics:")
print(res[res.metric.isin(["AUROC", "AUPRC"])][["metric", "group", "value"]].round(4).to_string(index=False))

joblib.dump(p, BASE + f"/prep_seed{SEED}.joblib")
np.savez(BASE + f"/split_seed{SEED}.npz", **split)
ledger = dict(
    seed=SEED, n_train=len(split["train"]), n_val=len(split["val"]), n_cal=len(split["cal"]),
    removed_from_input=["FLOW_START_MILLISECONDS (kept only for grouping)", "FLOW_END_MILLISECONDS",
                        "IPV4_SRC_ADDR", "IPV4_DST_ADDR", "L4_SRC_PORT", "L4_DST_PORT",
                        "DNS_QUERY_ID", "Label", "Attack"],
    numeric_features=p["num_keep"], onehot_features=list(p["ohe"].get_feature_names_out()),
    log1p_columns=p["log_cols"], dropped_constant=p["dropped_const"],
    dropped_correlated=p["dropped_corr"],
    versions=dict(python=sys.version.split()[0], pandas=pd.__version__,
                  numpy=np.__version__, sklearn=sklearn.__version__))
json.dump(ledger, open(BASE + f"/audit_ledger_seed{SEED}.json", "w"), indent=2)
print("\nSaved results, prep, split and audit ledger to", BASE)


partition sizes: {'train': 355336, 'val': 60000, 'cal': 185000}
ALL AUDIT ASSERTIONS PASSED
numeric features kept: 35
dropped (constant): []
dropped (correlated): ['MAX_TTL', 'MIN_IP_PKT_LEN', 'MAX_IP_PKT_LEN', 'DST_TO_SRC_AVG_THROUGHPUT', 'ICMP_IPV4_TYPE', 'SRC_TO_DST_IAT_MAX', 'SRC_TO_DST_IAT_AVG', 'DST_TO_SRC_IAT_MAX', 'DST_TO_SRC_IAT_AVG']
feature matrix: (3019057, 45)
scoring done in 0.3 min

FPR and recall (columns = nominal false-alarm target):
alpha                          0.001   0.010   0.050
metric group                                        
FPR    benign_2019-04-24      0.0019  0.0286  0.0988
       benign_2019-04-25      0.0063  0.0303  0.2184
       benign_2019-04-26      0.0046  0.0326  0.2032
       benign_2019-04-27      0.0519  0.4826  0.6884
       benign_2019-04-28      0.1818  0.4241  0.8716
       benign_2019-04-29      0.0010  0.0078  0.1070
       benign_later_all       0.0344  0.1551  0.3398
       val_benign_2019-04-23  0.0017  0.0115  0.0571
recall ddos   

In [ ]:
import numpy as np, pandas as pd, sys, time, joblib, torch, torch.nn as nn
sys.path.append("/content/drive/MyDrive/nfton")
import importlib, common
importlib.reload(common)
from common import *
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM

SEED = 13
BASE = "/content/drive/MyDrive/nfton"
np.random.seed(SEED)

ws = pd.read_parquet(BASE + "/working_set.parquet").reset_index(drop=True)
split = make_split(ws, SEED)
p = fit_prep(ws.loc[split["train"]])
print(audit(ws, split, p))
X = apply_prep(p, ws)
D = X.shape[1]
print("feature matrix:", X.shape)

scan23 = ws.index[(ws["day"] == DEV_DAY) & (ws["Attack"] == "scanning")].to_numpy()
need = np.setdiff1d(ws.index.to_numpy(), np.r_[split["train"], scan23])

rng = np.random.default_rng(SEED)
enc_idx = rng.choice(split["train"], size=min(100000, len(split["train"])), replace=False)
svm_idx = rng.choice(split["train"], size=min(25000, len(split["train"])), replace=False)
if_idx  = rng.choice(split["train"], size=min(50000, len(split["train"])), replace=False)

class AE(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(d,128), nn.ReLU(), nn.Linear(128,64), nn.ReLU(), nn.Linear(64,32))
        self.dec = nn.Sequential(nn.Linear(32,64), nn.ReLU(), nn.Linear(64,128), nn.ReLU(), nn.Linear(128,d))
    def forward(self, x):
        return self.dec(self.enc(x))

def train_ae(Xtr, Xva, mask_rate, seed, max_epochs=40, patience=5):
    torch.manual_seed(seed)
    g = torch.Generator().manual_seed(seed)
    model = AE(Xtr.shape[1])
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    Xt, Xv = torch.from_numpy(Xtr), torch.from_numpy(Xva)
    best, best_state, wait = np.inf, None, 0
    for ep in range(max_epochs):
        model.train()
        perm = torch.randperm(len(Xt), generator=g)
        for i in range(0, len(perm), 512):
            xb = Xt[perm[i:i+512]]
            xin = xb * (torch.rand(xb.shape, generator=g) >= mask_rate).float() if mask_rate > 0 else xb
            loss = ((model(xin) - xb) ** 2).mean()
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            vl = ((model(Xv) - Xv) ** 2).mean().item()
        if vl < best - 1e-6:
            best, wait = vl, 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience:
                break
    model.load_state_dict(best_state)
    model.eval()
    print(f"  autoencoder stopped after {ep+1} epochs, best validation loss {best:.4f}")
    return model

def encode(model, Xpart):
    with torch.no_grad():
        return model.enc(torch.from_numpy(Xpart)).numpy()

def score_all(fn, chunk=50000):
    sc = pd.Series(np.nan, index=ws.index)
    for i in range(0, len(need), chunk):
        part = need[i:i+chunk]
        sc.loc[part] = fn(part)
    return sc

results, all_scores = [], {}

# 1) Isolation Forest
t0 = time.time()
iforest = IsolationForest(n_estimators=200, random_state=SEED, n_jobs=-1).fit(X[if_idx])
all_scores["IsolationForest"] = score_all(lambda part: -iforest.score_samples(X[part]))
print("IsolationForest done: %.1f min" % ((time.time()-t0)/60))

# 2) Proposed: denoising MLP autoencoder + OC-SVM on 32-d embeddings
t0 = time.time()
ae = train_ae(X[enc_idx], X[split["val"]], mask_rate=0.10, seed=SEED)
torch.save(ae.state_dict(), BASE + f"/ae_mask10_seed{SEED}.pt")
svm_emb = OneClassSVM(kernel="rbf", nu=0.01, gamma="scale").fit(encode(ae, X[svm_idx]))
print("  embedding OC-SVM support vectors:", len(svm_emb.support_))
all_scores["MLP_OCSVM"] = score_all(lambda part: -svm_emb.decision_function(encode(ae, X[part])))
print("MLP_OCSVM done: %.1f min" % ((time.time()-t0)/60))

# 3) Raw-feature OC-SVM
t0 = time.time()
svm_raw = OneClassSVM(kernel="rbf", nu=0.01, gamma="scale").fit(X[svm_idx])
print("  raw OC-SVM support vectors:", len(svm_raw.support_))
all_scores["RawOCSVM"] = score_all(lambda part: -svm_raw.decision_function(X[part]))
print("RawOCSVM done: %.1f min" % ((time.time()-t0)/60))

for name, sc in all_scores.items():
    r = evaluate(sc, ws, split, name, SEED)
    r.to_csv(BASE + f"/results_{name}_seed{SEED}.csv", index=False)
    np.save(BASE + f"/scores_{name}_seed{SEED}.npy", sc.to_numpy().astype("float32"))
    results.append(r)
joblib.dump(dict(svm_emb=svm_emb, svm_raw=svm_raw), BASE + f"/ocsvm_seed{SEED}.joblib")

allres = pd.concat(results)
print("\nAt nominal alpha = 1% (FPR and recall):")
print(allres[(allres.alpha == 0.01) & allres.metric.isin(["FPR", "recall"])]
      .pivot(index=["metric", "group"], columns="model", values="value").round(4).to_string())
print("\nRanking metrics:")
print(allres[allres.metric.isin(["AUROC", "AUPRC"])]
      .pivot(index=["metric", "group"], columns="model", values="value").round(4).to_string())
print("\nSaved results and scores for seed", SEED)

ALL AUDIT ASSERTIONS PASSED
feature matrix: (3019057, 45)
IsolationForest done: 0.2 min
  autoencoder stopped after 22 epochs, best validation loss 0.0103
  embedding OC-SVM support vectors: 319
MLP_OCSVM done: 1.0 min
  raw OC-SVM support vectors: 335
RawOCSVM done: 0.9 min

At nominal alpha = 1% (FPR and recall):
model                         IsolationForest  MLP_OCSVM  RawOCSVM
metric group                                                      
FPR    benign_2019-04-24               0.0321     0.0491    0.0491
       benign_2019-04-25               0.0355     0.4397    0.5603
       benign_2019-04-26               0.0410     0.3608    0.5129
       benign_2019-04-27               0.5034     0.5954    0.7481
       benign_2019-04-28               0.4560     0.7095    0.7179
       benign_2019-04-29               0.0107     0.0991    0.0988
       benign_later_all                0.1663     0.3592    0.4346
       val_benign_2019-04-23           0.0096     0.0097    0.0100
recall ddos  

In [ ]:
import numpy as np, pandas as pd, sys, time, joblib, torch, torch.nn as nn, os, gc, glob
import lightgbm as lgb
sys.path.append("/content/drive/MyDrive/nfton")
import importlib, common
importlib.reload(common)
from common import *
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM

BASE = "/content/drive/MyDrive/nfton"
SEEDS = [13, 23, 37, 51, 73]
print("lightgbm", lgb.__version__, "| torch", torch.__version__)

ws = pd.read_parquet(BASE + "/working_set.parquet").reset_index(drop=True)
scan23 = ws.index[(ws["day"] == DEV_DAY) & (ws["Attack"] == "scanning")].to_numpy()

class AE(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(d,128), nn.ReLU(), nn.Linear(128,64), nn.ReLU(), nn.Linear(64,32))
        self.dec = nn.Sequential(nn.Linear(32,64), nn.ReLU(), nn.Linear(64,128), nn.ReLU(), nn.Linear(128,d))
    def forward(self, x):
        return self.dec(self.enc(x))

def train_ae(Xtr, Xva, mask_rate, seed, max_epochs=40, patience=5):
    torch.manual_seed(seed)
    g = torch.Generator().manual_seed(seed)
    model = AE(Xtr.shape[1])
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    Xt, Xv = torch.from_numpy(Xtr), torch.from_numpy(Xva)
    best, best_state, wait = np.inf, None, 0
    for ep in range(max_epochs):
        model.train()
        perm = torch.randperm(len(Xt), generator=g)
        for i in range(0, len(perm), 512):
            xb = Xt[perm[i:i+512]]
            xin = xb * (torch.rand(xb.shape, generator=g) >= mask_rate).float() if mask_rate > 0 else xb
            loss = ((model(xin) - xb) ** 2).mean()
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            vl = ((model(Xv) - Xv) ** 2).mean().item()
        if vl < best - 1e-6:
            best, wait = vl, 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience:
                break
    model.load_state_dict(best_state); model.eval()
    print(f"  autoencoder (mask {mask_rate}) stopped after {ep+1} epochs, best val loss {best:.4f}")
    return model

def encode(model, Xpart):
    with torch.no_grad():
        return model.enc(torch.from_numpy(Xpart)).numpy()

for SEED in SEEDS:
    flag = BASE + f"/done_seed{SEED}.flag"
    if os.path.exists(flag):
        print(f"seed {SEED}: already done, skipping"); continue
    print(f"\n===== SEED {SEED} =====")
    T0 = time.time()
    np.random.seed(SEED)
    split = make_split(ws, SEED)
    p = fit_prep(ws.loc[split["train"]])
    print(audit(ws, split, p))
    X = apply_prep(p, ws)
    need = np.setdiff1d(ws.index.to_numpy(), np.r_[split["train"], scan23])

    def score_all(fn, chunk=50000):
        sc = pd.Series(np.nan, index=ws.index)
        for i in range(0, len(need), chunk):
            part = need[i:i+chunk]
            sc.loc[part] = fn(part)
        return sc

    rng = np.random.default_rng(SEED)
    enc_idx = rng.choice(split["train"], size=min(100000, len(split["train"])), replace=False)
    svm_idx = rng.choice(split["train"], size=min(25000, len(split["train"])), replace=False)
    if_idx  = rng.choice(split["train"], size=min(50000, len(split["train"])), replace=False)

    all_scores = {}
    iforest = IsolationForest(n_estimators=200, random_state=SEED, n_jobs=-1).fit(X[if_idx])
    all_scores["IsolationForest"] = score_all(lambda part: -iforest.score_samples(X[part]))

    ae10 = train_ae(X[enc_idx], X[split["val"]], 0.10, SEED)
    svm10 = OneClassSVM(kernel="rbf", nu=0.01, gamma="scale").fit(encode(ae10, X[svm_idx]))
    all_scores["MLP_OCSVM"] = score_all(lambda part: -svm10.decision_function(encode(ae10, X[part])))
    torch.save(ae10.state_dict(), BASE + f"/ae_mask10_seed{SEED}.pt")

    ae0 = train_ae(X[enc_idx], X[split["val"]], 0.0, SEED)
    svm0 = OneClassSVM(kernel="rbf", nu=0.01, gamma="scale").fit(encode(ae0, X[svm_idx]))
    all_scores["MLP_OCSVM_mask0"] = score_all(lambda part: -svm0.decision_function(encode(ae0, X[part])))
    torch.save(ae0.state_dict(), BASE + f"/ae_mask0_seed{SEED}.pt")

    svm_raw = OneClassSVM(kernel="rbf", nu=0.01, gamma="scale").fit(X[svm_idx])
    all_scores["RawOCSVM"] = score_all(lambda part: -svm_raw.decision_function(X[part]))

    # LightGBM supervised reference: benign train + 23 Apr scanning only
    scan_sorted = scan23[np.argsort(ws.loc[scan23, "FLOW_START_MILLISECONDS"].to_numpy())]
    sb = np.arange(len(scan_sorted)) // BLOCK
    order = np.random.default_rng(SEED).permutation(int(sb.max()) + 1)
    val_blocks = order[:max(1, int(round(0.2 * len(order))))]
    is_val = np.isin(sb, val_blocks)
    s_tr, s_va = scan_sorted[~is_val], scan_sorted[is_val]
    Xl_tr = np.vstack([X[split["train"]], X[s_tr]])
    yl_tr = np.r_[np.zeros(len(split["train"])), np.ones(len(s_tr))]
    Xl_va = np.vstack([X[split["val"]], X[s_va]])
    yl_va = np.r_[np.zeros(len(split["val"])), np.ones(len(s_va))]
    clf = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=31,
                             subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                             random_state=SEED, n_jobs=-1, verbose=-1)
    clf.fit(Xl_tr, yl_tr, eval_set=[(Xl_va, yl_va)],
            callbacks=[lgb.early_stopping(30, verbose=False)])
    print("  LightGBM best iteration:", clf.best_iteration_)
    all_scores["LightGBM"] = score_all(lambda part: clf.predict(X[part], raw_score=True))

    for name, sc in all_scores.items():
        r = evaluate(sc, ws, split, name, SEED)
        r.to_csv(BASE + f"/results_{name}_seed{SEED}.csv", index=False)
        np.save(BASE + f"/scores_{name}_seed{SEED}.npy", sc.to_numpy().astype("float32"))
    np.savez(BASE + f"/split_seed{SEED}.npz", **split)
    joblib.dump(p, BASE + f"/prep_seed{SEED}.joblib")
    open(flag, "w").write("done")
    print("seed %d finished in %.1f min" % (SEED, (time.time() - T0) / 60))
    del X, Xl_tr, Xl_va, all_scores; gc.collect()

# ---------- summary across seeds ----------
files = glob.glob(BASE + "/results_*_seed*.csv")
df = pd.concat([pd.read_csv(f) for f in files])
print("\nseeds found per model:\n", df.groupby("model").seed.nunique().to_string())
d = df[((df.alpha == 0.01) & df.metric.isin(["FPR", "recall"]) & (df.group != "benign_later_all"))
       | ((df.metric == "AUROC") & (df.group == "dos+ddos"))]
agg = d.groupby(["metric", "group", "model"]).value.agg(["mean", "std"])
agg["txt"] = agg["mean"].round(3).astype(str) + " ± " + agg["std"].round(3).astype(str)
print("\nMean ± SD over seeds, nominal alpha = 1%:")
print(agg["txt"].unstack("model").to_string())

lightgbm 4.6.0 | torch 2.14.1+cpu

===== SEED 13 =====
ALL AUDIT ASSERTIONS PASSED
  autoencoder (mask 0.1) stopped after 22 epochs, best val loss 0.0103
  autoencoder (mask 0.0) stopped after 38 epochs, best val loss 0.0025
  LightGBM best iteration: 240


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/ut

seed 13 finished in 4.5 min

===== SEED 23 =====
ALL AUDIT ASSERTIONS PASSED
  autoencoder (mask 0.1) stopped after 15 epochs, best val loss 0.0222
  autoencoder (mask 0.0) stopped after 18 epochs, best val loss 0.0102
  LightGBM best iteration: 53


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/ut

seed 23 finished in 3.4 min

===== SEED 37 =====
ALL AUDIT ASSERTIONS PASSED
  autoencoder (mask 0.1) stopped after 35 epochs, best val loss 0.0078
  autoencoder (mask 0.0) stopped after 28 epochs, best val loss 0.0041
  LightGBM best iteration: 65


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/ut

seed 37 finished in 3.8 min

===== SEED 51 =====
ALL AUDIT ASSERTIONS PASSED
  autoencoder (mask 0.1) stopped after 40 epochs, best val loss 0.0098
  autoencoder (mask 0.0) stopped after 25 epochs, best val loss 0.0042
  LightGBM best iteration: 107


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/ut

seed 51 finished in 3.9 min

===== SEED 73 =====
ALL AUDIT ASSERTIONS PASSED
  autoencoder (mask 0.1) stopped after 34 epochs, best val loss 0.0145
  autoencoder (mask 0.0) stopped after 30 epochs, best val loss 0.0071
  LightGBM best iteration: 34


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/ut

seed 73 finished in 3.8 min

seeds found per model:
 model
IsolationForest    5
LightGBM           5
MLP_OCSVM          5
MLP_OCSVM_mask0    5
RawOCSVM           5

Mean ± SD over seeds, nominal alpha = 1%:
model                        IsolationForest       LightGBM      MLP_OCSVM MLP_OCSVM_mask0       RawOCSVM
metric group                                                                                             
AUROC  dos+ddos                0.695 ± 0.012  0.471 ± 0.081  0.587 ± 0.018   0.594 ± 0.026  0.639 ± 0.015
FPR    benign_2019-04-24       0.034 ± 0.002  0.001 ± 0.002  0.049 ± 0.004   0.049 ± 0.002   0.05 ± 0.004
       benign_2019-04-25         0.04 ± 0.01      0.0 ± 0.0   0.428 ± 0.06   0.423 ± 0.029  0.491 ± 0.075
       benign_2019-04-26        0.047 ± 0.01      0.0 ± 0.0  0.372 ± 0.071   0.364 ± 0.032  0.443 ± 0.085
       benign_2019-04-27       0.505 ± 0.013      0.0 ± 0.0   0.612 ± 0.04   0.605 ± 0.025   0.664 ± 0.07
       benign_2019-04-28       0.501 ± 0.067      0

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import numpy as np, pandas as pd, sys, warnings
warnings.filterwarnings("ignore")
sys.path.append("/content/drive/MyDrive/nfton")
from common import *
from scipy.stats import ks_2samp

BASE = "/content/drive/MyDrive/nfton"
SEEDS = [13, 23, 37, 51, 73]
MODELS = ["IsolationForest", "MLP_OCSVM", "RawOCSVM"]
WIN, MIN_WIN, KS_T, PSI_T = 5000, 500, 0.10, 0.25

ws = pd.read_parquet(BASE + "/working_set.parquet").reset_index(drop=True)
day = ws["day"].to_numpy(); att = ws["Attack"].to_numpy()
tms = ws["FLOW_START_MILLISECONDS"].to_numpy()

def psi(ref, cur):
    edges = np.unique(np.quantile(ref, np.linspace(0, 1, 11)))
    edges[0], edges[-1] = -np.inf, np.inf
    pr = np.histogram(ref, edges)[0] / len(ref)
    pc = np.histogram(cur, edges)[0] / len(cur)
    pr, pc = np.clip(pr, 1e-4, None), np.clip(pc, 1e-4, None)
    return float(np.sum((pc - pr) * np.log(pc / pr)))

rows = []
for seed in SEEDS:
    split = make_split(ws, seed)
    for model in MODELS:
        s = np.load(BASE + f"/scores_{model}_seed{seed}.npy")
        ref = s[split["cal"]]
        tau1 = conformal_threshold(ref, 0.01)
        for d in sorted(set(day[(att == "Benign") & (day >= "2019-04-24")])):
            idx = np.where((att == "Benign") & (day == d))[0]
            idx = idx[np.argsort(tms[idx])]
            for w, i in enumerate(range(0, len(idx), WIN)):
                part = idx[i:i + WIN]
                if len(part) < MIN_WIN: continue
                cur = s[part]
                D = ks_2samp(ref, cur).statistic
                P = psi(ref, cur)
                rows.append(dict(seed=seed, model=model, day=d, window=w, n=len(part),
                                 KS_D=D, PSI=P, trig=(D >= KS_T) or (P >= PSI_T),
                                 FPR_1pct=(cur > tau1).mean()))
mon = pd.DataFrame(rows)
mon.to_csv(BASE + "/monitor_benign_windows.csv", index=False)

summ = (mon.groupby(["model", "day"])
          .agg(windows=("trig", "size"), trigger_rate=("trig", "mean"),
               median_KS_D=("KS_D", "median"), median_PSI=("PSI", "median"),
               mean_FPR=("FPR_1pct", "mean")).round(3))
print(summ.to_string())

                            windows  trigger_rate  median_KS_D  median_PSI  mean_FPR
model           day                                                                 
IsolationForest 2019-04-24      300          0.98        0.298       1.338     0.034
                2019-04-25      300          1.00        0.903       8.952     0.040
                2019-04-26      300          1.00        0.903       8.894     0.047
                2019-04-27      300          1.00        0.911       8.682     0.505
                2019-04-28      220          1.00        0.937       8.901     0.498
                2019-04-29      300          1.00        0.945       9.177     0.017
MLP_OCSVM       2019-04-24      300          1.00        0.256       1.021     0.049
                2019-04-25      300          1.00        0.510       2.659     0.428
                2019-04-26      300          1.00        0.513       2.951     0.373
                2019-04-27      300          1.00        0.629   

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import numpy as np, pandas as pd, sys, warnings
warnings.filterwarnings("ignore")
sys.path.append("/content/drive/MyDrive/nfton")
from common import *
from scipy.stats import ks_2samp

BASE = "/content/drive/MyDrive/nfton"
SEEDS = [13, 23, 37, 51, 73]
MODELS = ["IsolationForest", "MLP_OCSVM", "RawOCSVM"]
WIN, MIN_WIN, KS_T, PSI_T = 5000, 500, 0.10, 0.25
MULTS = (1.5, 2.0, 3.0)

ws = pd.read_parquet(BASE + "/working_set.parquet").reset_index(drop=True)
day = ws["day"].to_numpy(); att = ws["Attack"].to_numpy()
tms = ws["FLOW_START_MILLISECONDS"].to_numpy()

def psi(ref, cur):
    edges = np.unique(np.quantile(ref, np.linspace(0, 1, 11)))
    edges[0], edges[-1] = -np.inf, np.inf
    pr = np.histogram(ref, edges)[0] / len(ref)
    pc = np.histogram(cur, edges)[0] / len(cur)
    pr, pc = np.clip(pr, 1e-4, None), np.clip(pc, 1e-4, None)
    return float(np.sum((pc - pr) * np.log(pc / pr)))

rows = []
for seed in SEEDS:
    split = make_split(ws, seed)
    groups = [("2019-04-23 (same-day val)", split["val"])]
    for d in sorted(set(day[(att == "Benign") & (day >= "2019-04-24")])):
        groups.append((d, np.where((att == "Benign") & (day == d))[0]))
    for model in MODELS:
        s = np.load(BASE + f"/scores_{model}_seed{seed}.npy")
        ref = s[split["cal"]]
        tau = conformal_threshold(ref, 0.01)
        for gname, idx in groups:
            idx = idx[np.argsort(tms[idx])]
            for w, i in enumerate(range(0, len(idx), WIN)):
                part = idx[i:i + WIN]
                if len(part) < MIN_WIN: continue
                cur = s[part]
                D, P = ks_2samp(ref, cur).statistic, psi(ref, cur)
                r = (cur > tau).mean()
                row = dict(seed=seed, model=model, day=gname, window=w, n=len(part),
                           KS_D=D, PSI=P, exceed_rate=r, trig_KSPSI=(D >= KS_T) or (P >= PSI_T))
                for m in MULTS:
                    row[f"trig_exc_{m}x"] = r >= m * 0.01
                rows.append(row)
mon = pd.DataFrame(rows)
mon.to_csv(BASE + "/monitor_windows_F1b.csv", index=False)

summ = (mon.groupby(["model", "day"])
          .agg(windows=("n", "size"),
               realized_FPR=("exceed_rate", "mean"),
               trig_KSPSI=("trig_KSPSI", "mean"),
               trig_exc_1_5x=("trig_exc_1.5x", "mean"),
               trig_exc_2x=("trig_exc_2.0x", "mean"),
               trig_exc_3x=("trig_exc_3.0x", "mean")).round(3))
print(summ.to_string())

                                           windows  realized_FPR  trig_KSPSI  trig_exc_1_5x  trig_exc_2x  trig_exc_3x
model           day                                                                                                  
IsolationForest 2019-04-23 (same-day val)       60         0.012       0.983          0.317        0.200        0.083
                2019-04-24                     300         0.034       0.980          0.450        0.390        0.307
                2019-04-25                     300         0.040       1.000          0.570        0.543        0.423
                2019-04-26                     300         0.047       1.000          0.480        0.437        0.220
                2019-04-27                     300         0.505       1.000          1.000        1.000        1.000
                2019-04-28                     220         0.498       1.000          1.000        1.000        1.000
                2019-04-29                     300      

In [ ]:
import numpy as np, pandas as pd, sys, warnings
warnings.filterwarnings("ignore")
sys.path.append("/content/drive/MyDrive/nfton")
from common import *

BASE = "/content/drive/MyDrive/nfton"
SEEDS = [13, 23, 37, 51, 73]
MODELS = ["IsolationForest", "MLP_OCSVM", "RawOCSVM"]
ALPHA, BATCH, WIN, MIN_WIN, MULT = 0.01, 20000, 5000, 500, 2.0
EPS = (0.0, 0.01)

ws = pd.read_parquet(BASE + "/working_set.parquet").reset_index(drop=True)
day = ws["day"].to_numpy(); att = ws["Attack"].to_numpy()
tms = ws["FLOW_START_MILLISECONDS"].to_numpy()
fam = np.where(np.isin(att, ["dos", "ddos"]), "dosfam", np.where(att == "xss", "xss", "other"))
days = sorted(set(day[day >= "2019-04-24"]))

def trimmed_threshold(sc, alpha, eps):
    s = np.sort(sc)
    if eps > 0:
        s = s[:int(np.floor(len(s) * (1 - eps)))]
    return conformal_threshold(s, alpha)

rows = []
for seed in SEEDS:
    split = make_split(ws, seed)
    for model in MODELS:
        s = np.load(BASE + f"/scores_{model}_seed{seed}.npy")
        tau0 = conformal_threshold(s[split["cal"]], ALPHA)
        for d in days:
            in_day = np.where(day == d)[0]
            b = in_day[att[in_day] == "Benign"]
            b = b[np.argsort(tms[b])]
            batch, ev = b[:BATCH], b[BATCH:]
            row = dict(seed=seed, model=model, day=d, n_eval=len(ev),
                       FPR_stale=(s[ev] > tau0).mean())
            dec = fa = nw = nab = 0
            for i in range(0, len(ev), WIN):
                part = ev[i:i + WIN]
                if len(part) < MIN_WIN: continue
                nw += 1
                if (s[part] > tau0).mean() >= MULT * ALPHA:
                    nab += 1
                else:
                    dec += len(part); fa += int((s[part] > tau0).sum())
            row["abstain_rate"] = nab / nw if nw else np.nan
            row["FPR_decided"] = fa / dec if dec > 0 else np.nan
            taus = {e: trimmed_threshold(s[batch], ALPHA, e) for e in EPS}
            for e in EPS:
                row[f"FPR_recal_eps{e}"] = (s[ev] > taus[e]).mean()
            for f in ("dosfam", "xss"):
                m = in_day[fam[in_day] == f]
                row[f"recall_stale_{f}"] = (s[m] > tau0).mean() if len(m) else np.nan
                for e in EPS:
                    row[f"recall_recal{e}_{f}"] = (s[m] > taus[e]).mean() if len(m) else np.nan
            rows.append(row)

res = pd.DataFrame(rows)
res.to_csv(BASE + "/E3_policies_per_seed.csv", index=False)
cols = ["FPR_stale", "abstain_rate", "FPR_decided", "FPR_recal_eps0.0", "FPR_recal_eps0.01"]
print("Realized benign FPR (target 0.01), mean over 5 seeds:")
print(res.groupby(["model", "day"])[cols].mean().round(4).to_string())
rc = [c for c in res.columns if c.startswith("recall_")]
print("\nAttack recall on that day's attack flows, mean over 5 seeds:")
print(res.groupby(["model", "day"])[rc].mean().round(3).dropna(how="all").to_string())

Realized benign FPR (target 0.01), mean over 5 seeds:
                            FPR_stale  abstain_rate  FPR_decided  FPR_recal_eps0.0  FPR_recal_eps0.01
model           day                                                                                  
IsolationForest 2019-04-24     0.0325        0.3679       0.0055            0.0051             0.0147
                2019-04-25     0.0254        0.5107       0.0102            0.0004             0.0013
                2019-04-26     0.0491        0.4393       0.0122            0.0422             0.0472
                2019-04-27     0.5113        1.0000          NaN            0.0422             0.0649
                2019-04-28     0.5143        1.0000          NaN            0.0175             0.0305
                2019-04-29     0.0041        0.0214       0.0005            0.0002             0.0003
MLP_OCSVM       2019-04-24     0.0501        0.3250       0.0097            0.0308             0.0399
                2019-04-25  

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
import os
print(os.path.exists("/content/drive/MyDrive/nfton/common.py"))
print(sorted(os.listdir("/content/drive/MyDrive/nfton"))[:15])

True
['E3_policies_per_seed.csv', '__pycache__', 'ae_mask0_seed13.pt', 'ae_mask0_seed23.pt', 'ae_mask0_seed37.pt', 'ae_mask0_seed51.pt', 'ae_mask0_seed73.pt', 'ae_mask10_seed13.pt', 'ae_mask10_seed23.pt', 'ae_mask10_seed37.pt', 'ae_mask10_seed51.pt', 'ae_mask10_seed73.pt', 'audit_ledger_seed13.json', 'common.py', 'data']


In [10]:
import numpy as np, pandas as pd, sys, warnings, importlib
warnings.filterwarnings("ignore")
P = "/content/drive/MyDrive/nfton"
importlib.invalidate_caches()
sys.path_importer_cache.pop(P, None)
if P not in sys.path:
    sys.path.insert(0, P)
from common import *

BASE = "/content/drive/MyDrive/nfton"
SEEDS = [13, 23, 37, 51, 73]
MODELS = ["IsolationForest", "MLP_OCSVM", "RawOCSVM"]
ALPHA, BATCH = 0.01, 20000

ws = pd.read_parquet(BASE + "/working_set.parquet").reset_index(drop=True)
day = ws["day"].to_numpy(); att = ws["Attack"].to_numpy()
tms = ws["FLOW_START_MILLISECONDS"].to_numpy()
isdos = np.isin(att, ["dos", "ddos"])

# ---------- Part A: score saturation (seed 13) ----------
print("PART A: share of flows whose score equals the model's maximum score (seed 13)")
split13 = make_split(ws, 13)
rowsA = []
for model in MODELS:
    s = np.load(BASE + f"/scores_{model}_seed13.npy")
    smax = np.nanmax(s)
    r = {"model": model,
         "tau_at_1pct": conformal_threshold(s[split13["cal"]], ALPHA),
         "max_score": smax,
         "calibration": (s[split13["cal"]] == smax).mean()}
    for d in sorted(set(day[day >= "2019-04-24"])):
        r["benign_" + d[5:]] = (s[(day == d) & (att == "Benign")] == smax).mean()
    r["dos_family"] = (s[isdos] == smax).mean()
    rowsA.append(r)
print(pd.DataFrame(rowsA).set_index("model").round(4).T.to_string())

# ---------- Part B: contaminated verified-benign batch ----------
rowsB = []
for seed in SEEDS:
    for model in MODELS:
        s = np.load(BASE + f"/scores_{model}_seed{seed}.npy")
        for d in ["2019-04-24", "2019-04-25", "2019-04-26"]:
            b = np.where((day == d) & (att == "Benign"))[0]
            b = b[np.argsort(tms[b])]
            clean, ev = b[:BATCH], b[BATCH:]
            a_idx = np.where((day == d) & isdos)[0]
            for c in (0.01, 0.05):
                k = int(c * BATCH)
                a_pick = np.random.default_rng(seed).choice(a_idx, size=k, replace=False)
                batch = np.r_[clean[:BATCH - k], a_pick]
                a_eval = np.setdiff1d(a_idx, a_pick)
                for eps in (0.0, 0.01, 0.05):
                    sc = np.sort(s[batch])
                    if eps > 0:
                        sc = sc[:int(np.floor(len(sc) * (1 - eps)))]
                    tau = conformal_threshold(sc, ALPHA)
                    rowsB.append(dict(seed=seed, model=model, day=d, contamination=c, eps=eps,
                                      FPR=(s[ev] > tau).mean(), recall=(s[a_eval] > tau).mean()))
B = pd.DataFrame(rowsB)
B.to_csv(BASE + "/E3b_contaminated_batch.csv", index=False)
for m in ("FPR", "recall"):
    print(f"\nPART B: {m} (mean over 5 seeds); columns = trimming eps; target FPR = 0.01")
    print(B.pivot_table(index=["model", "day", "contamination"], columns="eps",
                        values=m, aggfunc="mean").round(4).to_string())

PART A: share of flows whose score equals the model's maximum score (seed 13)
model         IsolationForest  MLP_OCSVM  RawOCSVM
tau_at_1pct            0.7276     0.0246    0.0528
max_score              0.8071     2.9812    2.1734
calibration            0.0000     0.0001    0.0001
benign_04-24           0.0000     0.0012    0.0013
benign_04-25           0.0000     0.0208    0.0234
benign_04-26           0.0000     0.0335    0.0338
benign_04-27           0.0000     0.0117    0.0121
benign_04-28           0.0000     0.0134    0.0134
benign_04-29           0.0000     0.0151    0.0192
dos_family             0.0000     0.0322    0.0327

PART B: FPR (mean over 5 seeds); columns = trimming eps; target FPR = 0.01
eps                                         0.00    0.01    0.05
model           day        contamination                        
IsolationForest 2019-04-24 0.01           0.0051  0.0146  0.0329
                           0.05           0.0051  0.0140  0.0330
                2019-04-2

In [11]:
import numpy as np, pandas as pd, sys, warnings, importlib, itertools
from sklearn.metrics import roc_curve
warnings.filterwarnings("ignore")
P = "/content/drive/MyDrive/nfton"
importlib.invalidate_caches(); sys.path_importer_cache.pop(P, None)
if P not in sys.path: sys.path.insert(0, P)
from common import *

BASE = P
SEEDS = [13, 23, 37, 51, 73]
MODELS = ["IsolationForest", "MLP_OCSVM", "RawOCSVM"]
CONFIGS = ["IsolationForest", "MLP_OCSVM", "MLP_OCSVM_mask0", "RawOCSVM"]
ALPHA = 0.01

ws = pd.read_parquet(BASE + "/working_set.parquet").reset_index(drop=True)
day = ws["day"].to_numpy(); att = ws["Attack"].to_numpy()
later_b = (att == "Benign") & (day >= "2019-04-24")
dosfam = np.isin(att, ["dos", "ddos"])
y_all = np.r_[np.zeros(later_b.sum()), np.ones(dosfam.sum())]

rowsL2, rowsL3, rowsK = [], [], []
for seed in SEEDS:
    split = make_split(ws, seed)
    sc_, clean = {}, {}
    for name in CONFIGS:
        s = np.load(f"{BASE}/scores_{name}_seed{seed}.npy")
        sc_[name] = s
        tau = conformal_threshold(s[split["cal"]], ALPHA)
        clean[name] = ((s[dosfam] > tau).mean(), (s[later_b] > tau).mean())
    # L2: threshold chosen to maximise TEST balanced accuracy (uses test labels)
    for name in MODELS:
        s = sc_[name]
        fpr_a, tpr_a, thr = roc_curve(y_all, np.r_[s[later_b], s[dosfam]])
        ba = (tpr_a + 1 - fpr_a) / 2
        i = int(np.argmax(ba))
        rc, fc = clean[name]
        rowsL2.append(dict(seed=seed, model=name, clean_recall=rc, clean_FPR=fc,
                           clean_BA=(rc + 1 - fc) / 2, leak_recall=tpr_a[i],
                           leak_FPR=fpr_a[i], leak_BA=ba[i]))
    # L3: choose the configuration with the best held-out DoS-family recall
    recs = {n: clean[n][0] for n in CONFIGS}
    best = max(CONFIGS, key=lambda n: recs[n])
    rowsL3.append(dict(seed=seed, selected=best,
                       clean_recall=clean["MLP_OCSVM"][0], clean_FPR=clean["MLP_OCSVM"][1],
                       leak_recall=clean[best][0], leak_FPR=clean[best][1]))
    for k in range(1, len(CONFIGS) + 1):
        mx = [max(recs[n] for n in c) for c in itertools.combinations(CONFIGS, k)]
        rowsK.append(dict(seed=seed, k=k, mean_best_recall=np.mean(mx)))

L2, L3, K = pd.DataFrame(rowsL2), pd.DataFrame(rowsL3), pd.DataFrame(rowsK)
L2.to_csv(BASE + "/E5_L2.csv", index=False)
L3.to_csv(BASE + "/E5_L3.csv", index=False)
K.to_csv(BASE + "/E5_L3_best_of_k.csv", index=False)

c2 = ["clean_recall", "clean_FPR", "clean_BA", "leak_recall", "leak_FPR", "leak_BA"]
print("L2 (test-label threshold), mean over 5 seeds (DoS family vs pooled later benign):")
print(L2.groupby("model")[c2].mean().round(4).to_string())
print("\nL2 SD over seeds:")
print(L2.groupby("model")[c2].std().round(4).to_string())
print("\nL3 (config picked on held-out DoS recall), per seed:")
print(L3.round(4).to_string(index=False))
print("\nL3 mean: clean (predeclared MLP_OCSVM) vs leaky-selected")
print(L3[["clean_recall", "clean_FPR", "leak_recall", "leak_FPR"]].mean().round(4).to_string())
print("\nInflation vs number of configurations tried (mean best recall, k = pool size):")
print(K.groupby("k").mean_best_recall.agg(["mean", "std"]).round(4).to_string())

L2 (test-label threshold), mean over 5 seeds (DoS family vs pooled later benign):
                 clean_recall  clean_FPR  clean_BA  leak_recall  leak_FPR  leak_BA
model                                                                             
IsolationForest        0.0694     0.1755    0.4470       0.8684    0.3294   0.7695
MLP_OCSVM              0.4208     0.3699    0.5255       0.6490    0.3761   0.6365
RawOCSVM               0.7157     0.4014    0.6571       0.7087    0.3486   0.6801

L2 SD over seeds:
                 clean_recall  clean_FPR  clean_BA  leak_recall  leak_FPR  leak_BA
model                                                                             
IsolationForest        0.0036     0.0156    0.0067       0.0637    0.0174   0.0245
MLP_OCSVM              0.1399     0.0376    0.0517       0.2019    0.1798   0.0190
RawOCSVM               0.0144     0.0390    0.0195       0.0320    0.0399   0.0074

L3 (config picked on held-out DoS recall), per seed:
 seed selected 

In [12]:
import numpy as np, pandas as pd, sys, warnings, importlib, os, time, gc, glob, torch, torch.nn as nn
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
warnings.filterwarnings("ignore")
P = "/content/drive/MyDrive/nfton"
importlib.invalidate_caches(); sys.path_importer_cache.pop(P, None)
if P not in sys.path: sys.path.insert(0, P)
from common import *

BASE = P
SEEDS = [13, 23, 37, 51, 73]
ws = pd.read_parquet(BASE + "/working_set.parquet").reset_index(drop=True)
day = ws["day"].to_numpy()
scan23 = ws.index[(ws["day"] == DEV_DAY) & (ws["Attack"] == "scanning")].to_numpy()

class AE(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(d,128), nn.ReLU(), nn.Linear(128,64), nn.ReLU(), nn.Linear(64,32))
        self.dec = nn.Sequential(nn.Linear(32,64), nn.ReLU(), nn.Linear(64,128), nn.ReLU(), nn.Linear(128,d))
    def forward(self, x):
        return self.dec(self.enc(x))

def train_ae(Xtr, Xva, mask_rate, seed, max_epochs=40, patience=5):
    torch.manual_seed(seed)
    g = torch.Generator().manual_seed(seed)
    model = AE(Xtr.shape[1])
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    Xt, Xv = torch.from_numpy(Xtr), torch.from_numpy(Xva)
    best, best_state, wait = np.inf, None, 0
    for ep in range(max_epochs):
        model.train()
        perm = torch.randperm(len(Xt), generator=g)
        for i in range(0, len(perm), 512):
            xb = Xt[perm[i:i+512]]
            xin = xb * (torch.rand(xb.shape, generator=g) >= mask_rate).float() if mask_rate > 0 else xb
            loss = ((model(xin) - xb) ** 2).mean()
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            vl = ((model(Xv) - Xv) ** 2).mean().item()
        if vl < best - 1e-6:
            best, wait = vl, 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience: break
    model.load_state_dict(best_state); model.eval()
    return model

def encode(model, Xpart):
    with torch.no_grad():
        return model.enc(torch.from_numpy(Xpart)).numpy()

for SEED in SEEDS:
    if os.path.exists(BASE + f"/results_L1_MLP_OCSVM_seed{SEED}.csv"):
        print(f"seed {SEED}: already done, skipping"); continue
    T0 = time.time(); np.random.seed(SEED)
    split = make_split(ws, SEED)
    # --- the deliberate leak: preprocessing fitted on development + sampled test-period rows ---
    test_idx = np.where(day >= "2019-04-24")[0]
    leak_rows = np.random.default_rng(SEED).choice(test_idx, size=600000, replace=False)
    p = fit_prep(ws.loc[np.r_[split["train"], leak_rows]])
    print(f"seed {SEED}: LEAKY preprocessing kept {len(p['num_keep'])} numeric features; "
          f"dropped correlated: {len(p['dropped_corr'])}")
    X = apply_prep(p, ws)
    need = np.setdiff1d(ws.index.to_numpy(), np.r_[split["train"], scan23])

    def score_all(fn, chunk=50000):
        sc = pd.Series(np.nan, index=ws.index)
        for i in range(0, len(need), chunk):
            part = need[i:i+chunk]
            sc.loc[part] = fn(part)
        return sc

    rng = np.random.default_rng(SEED)   # same draw order as the clean runs
    enc_idx = rng.choice(split["train"], size=min(100000, len(split["train"])), replace=False)
    svm_idx = rng.choice(split["train"], size=min(25000, len(split["train"])), replace=False)
    if_idx  = rng.choice(split["train"], size=min(50000, len(split["train"])), replace=False)

    iforest = IsolationForest(n_estimators=200, random_state=SEED, n_jobs=-1).fit(X[if_idx])
    s_if = score_all(lambda part: -iforest.score_samples(X[part]))
    ae = train_ae(X[enc_idx], X[split["val"]], 0.10, SEED)
    svm = OneClassSVM(kernel="rbf", nu=0.01, gamma="scale").fit(encode(ae, X[svm_idx]))
    s_mlp = score_all(lambda part: -svm.decision_function(encode(ae, X[part])))

    evaluate(s_if, ws, split, "L1_IsolationForest", SEED).to_csv(
        BASE + f"/results_L1_IsolationForest_seed{SEED}.csv", index=False)
    evaluate(s_mlp, ws, split, "L1_MLP_OCSVM", SEED).to_csv(
        BASE + f"/results_L1_MLP_OCSVM_seed{SEED}.csv", index=False)
    print("seed %d finished in %.1f min" % (SEED, (time.time() - T0) / 60))
    del X; gc.collect()

# ---------- summary: clean vs L1 leak, nominal alpha = 1% ----------
def load(prefix):
    return pd.concat([pd.read_csv(f) for f in sorted(glob.glob(f"{BASE}/results_{prefix}_seed*.csv"))])
rows = []
for clean, leak in [("IsolationForest", "L1_IsolationForest"), ("MLP_OCSVM", "L1_MLP_OCSVM")]:
    for tag, name in (("clean", clean), ("L1 leak", leak)):
        d = load(name); d = d[d.seed.isin(SEEDS)]
        r = d[(d.metric == "recall") & (d.group == "dos+ddos") & (d.alpha == 0.01)].set_index("seed").value
        f = d[(d.metric == "FPR") & (d.group == "benign_later_all") & (d.alpha == 0.01)].set_index("seed").value
        a = d[(d.metric == "AUROC") & (d.group == "dos+ddos")].set_index("seed").value
        rows.append(dict(model=clean, protocol=tag,
                         recall=f"{r.mean():.3f} ± {r.std():.3f}",
                         FPR_later=f"{f.mean():.3f} ± {f.std():.3f}",
                         AUROC=f"{a.mean():.3f} ± {a.std():.3f}"))
print("\nL1 (preprocessing leak) vs clean, mean ± SD over seeds (alpha = 1%):")
print(pd.DataFrame(rows).to_string(index=False))

seed 13: LEAKY preprocessing kept 40 numeric features; dropped correlated: 4
seed 13 finished in 3.3 min
seed 23: LEAKY preprocessing kept 40 numeric features; dropped correlated: 4
seed 23 finished in 3.2 min
seed 37: LEAKY preprocessing kept 41 numeric features; dropped correlated: 3
seed 37 finished in 3.4 min
seed 51: LEAKY preprocessing kept 41 numeric features; dropped correlated: 3
seed 51 finished in 3.4 min
seed 73: LEAKY preprocessing kept 41 numeric features; dropped correlated: 3
seed 73 finished in 3.1 min

L1 (preprocessing leak) vs clean, mean ± SD over seeds (alpha = 1%):
          model protocol        recall     FPR_later         AUROC
IsolationForest    clean 0.069 ± 0.004 0.175 ± 0.016 0.695 ± 0.012
IsolationForest  L1 leak 0.068 ± 0.004 0.166 ± 0.010 0.688 ± 0.021
      MLP_OCSVM    clean 0.421 ± 0.140 0.370 ± 0.038 0.587 ± 0.018
      MLP_OCSVM  L1 leak 0.715 ± 0.004 0.368 ± 0.026 0.641 ± 0.019


In [13]:
import numpy as np, pandas as pd, sys, warnings, importlib
from scipy.stats import ks_2samp
warnings.filterwarnings("ignore")
P = "/content/drive/MyDrive/nfton"
importlib.invalidate_caches(); sys.path_importer_cache.pop(P, None)
if P not in sys.path: sys.path.insert(0, P)
from common import *

SEEDS = [13, 23, 37, 51, 73]
MODELS = ["IsolationForest", "MLP_OCSVM", "RawOCSVM"]
WIN, N_BEN, N_ATK, NW = 1000, 800, 200, 40
EPS_M, THR = 0.5, np.log(100.0)
KS_T, PSI_T = 0.10, 0.25
FAM = {"2019-04-24": ["dos"], "2019-04-25": ["dos", "ddos"],
       "2019-04-26": ["ddos"], "2019-04-27": ["xss"]}

ws = pd.read_parquet(P + "/working_set.parquet").reset_index(drop=True)
day = ws["day"].to_numpy(); att = ws["Attack"].to_numpy()
tms = ws["FLOW_START_MILLISECONDS"].to_numpy()

def psi(ref, cur):
    edges = np.unique(np.quantile(ref, np.linspace(0, 1, 11)))
    edges[0], edges[-1] = -np.inf, np.inf
    pr = np.histogram(ref, edges)[0] / len(ref)
    pc = np.histogram(cur, edges)[0] / len(cur)
    pr, pc = np.clip(pr, 1e-4, None), np.clip(pc, 1e-4, None)
    return float(np.sum((pc - pr) * np.log(pc / pr)))

rows = []
for seed in SEEDS:
    split = make_split(ws, seed)
    for model in MODELS:
        s = np.load(f"{P}/scores_{model}_seed{seed}.npy")
        ref = s[split["cal"]]; refs = np.sort(ref); n = len(refs)
        for d, fl in FAM.items():
            b = np.where((day == d) & (att == "Benign"))[0]
            b = b[np.argsort(tms[b])]
            a_idx = np.where((day == d) & np.isin(att, fl))[0]
            starts = np.linspace(0, len(b) - WIN, NW).astype(int)
            for st in starts:
                for kind in ("benign", "attack"):
                    if kind == "benign":
                        idx = b[st:st + WIN]
                    else:
                        r = np.random.default_rng([seed, int(d[-2:]), int(st)])
                        idx = np.r_[b[st:st + N_BEN], r.choice(a_idx, size=N_ATK, replace=False)]
                    x = s[idx]
                    p = (1 + (n - np.searchsorted(refs, x, side="left"))) / (n + 1)
                    lm = np.cumsum(np.log(EPS_M) + (EPS_M - 1) * np.log(p))
                    tail = bool((lm >= THR).any())
                    fc = int(np.argmax(lm >= THR)) if tail else -1
                    bulk = (ks_2samp(ref, x).statistic >= KS_T) or (psi(ref, x) >= PSI_T)
                    route = "alert" if tail else ("abstain" if bulk else "decide")
                    pre = (kind == "attack") and tail and fc < N_BEN
                    delay = (fc - N_BEN + 1) if (kind == "attack" and tail and fc >= N_BEN) else np.nan
                    rows.append(dict(seed=seed, model=model, day=d, kind=kind, route=route,
                                     cross_before_onset=pre, delay=delay))
R = pd.DataFrame(rows)
R.to_csv(P + "/H1_router_windows.csv", index=False)
for k in ("alert", "abstain", "decide"):
    R[k] = (R.route == k)
tab = R.groupby(["model", "day", "kind"]).agg(
    windows=("route", "size"), alert=("alert", "mean"), abstain=("abstain", "mean"),
    decide=("decide", "mean"), cross_before_onset=("cross_before_onset", "mean"),
    median_delay_flows=("delay", "median")).round(3)
print(tab.to_string())

                                   windows  alert  abstain  decide  cross_before_onset  median_delay_flows
model           day        kind                                                                           
IsolationForest 2019-04-24 attack      200  0.545    0.455   0.000               0.320               133.0
                           benign      200  0.320    0.665   0.015               0.000                 NaN
                2019-04-25 attack      200  1.000    0.000   0.000               1.000                 NaN
                           benign      200  1.000    0.000   0.000               0.000                 NaN
                2019-04-26 attack      200  1.000    0.000   0.000               1.000                 NaN
                           benign      200  1.000    0.000   0.000               0.000                 NaN
                2019-04-27 attack      200  1.000    0.000   0.000               1.000                 NaN
                           benign    

In [14]:
import numpy as np, pandas as pd, sys, warnings, importlib
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")
P = "/content/drive/MyDrive/nfton"
importlib.invalidate_caches(); sys.path_importer_cache.pop(P, None)
if P not in sys.path: sys.path.insert(0, P)
from common import *

SEEDS = [13, 23, 37, 51, 73]
M3 = ["IsolationForest", "MLP_OCSVM", "RawOCSVM"]
ALL5 = M3 + ["MLP_OCSVM_mask0", "LightGBM"]
B = 2000
ws = pd.read_parquet(P + "/working_set.parquet").reset_index(drop=True)
day = ws["day"].to_numpy(); att = ws["Attack"].to_numpy()
later_days = sorted(set(day[day >= "2019-04-24"]))

# ---------- (a) paired bootstrap CIs at alpha = 1% ----------
bits = np.array([[(k >> (2 - j)) & 1 for j in range(3)] for k in range(8)])
def boot(pid, rng):
    cnt = np.bincount(pid, minlength=8)
    return rng.multinomial(len(pid), cnt / len(pid), size=B) @ bits / len(pid)

rowsCI = []
for seed in SEEDS:
    split = make_split(ws, seed); rng = np.random.default_rng(seed)
    S = {m: np.load(f"{P}/scores_{m}_seed{seed}.npy") for m in M3}
    flag = np.stack([S[m] > conformal_threshold(S[m][split["cal"]], 0.01) for m in M3], axis=1).astype(int)
    pid = flag[:, 0] * 4 + flag[:, 1] * 2 + flag[:, 2]
    targets = {"recall_dos": att == "dos", "recall_ddos": att == "ddos", "recall_xss": att == "xss"}
    for d in later_days: targets["FPR_" + d] = (day == d) & (att == "Benign")
    for name, mask in targets.items():
        r = boot(pid[mask], rng)
        qs = {m: (flag[mask, j].mean(), r[:, j]) for j, m in enumerate(M3)}
        qs["MLP minus IF"] = (flag[mask, 1].mean() - flag[mask, 0].mean(), r[:, 1] - r[:, 0])
        qs["MLP minus Raw"] = (flag[mask, 1].mean() - flag[mask, 2].mean(), r[:, 1] - r[:, 2])
        for q, (est, dist) in qs.items():
            lo, hi = np.percentile(dist, [2.5, 97.5])
            rowsCI.append(dict(seed=seed, target=name, quantity=q, est=est, lo=lo, hi=hi))
C = pd.DataFrame(rowsCI); C.to_csv(P + "/bootstrap_CI_all_seeds.csv", index=False)
c13 = C[C.seed == 13].copy()
c13["txt"] = c13.apply(lambda r: f"{r.est:.3f} [{r.lo:.3f}, {r.hi:.3f}]", axis=1)
print("PAIRED BOOTSTRAP 95% CIs, seed 13, alpha = 1%:")
print(c13.pivot(index="target", columns="quantity", values="txt").to_string())
C["excludes_0"] = (C.lo > 0) | (C.hi < 0)
print("\nShare of the 5 seeds where the paired-difference CI excludes 0:")
print(C[C.quantity.str.contains("minus")].groupby(["target", "quantity"]).excludes_0.mean().unstack().round(2).to_string())

# ---------- (b) per-day AUROC (attack vs benign of the SAME day) ----------
dayfam = {"2019-04-24": att == "dos", "2019-04-25": np.isin(att, ["dos", "ddos"]),
          "2019-04-26": att == "ddos", "2019-04-27": att == "xss"}
rowsA = []
for seed in SEEDS:
    for m in ALL5:
        s = np.load(f"{P}/scores_{m}_seed{seed}.npy")
        for d, am in dayfam.items():
            a = am & (day == d); bm = (day == d) & (att == "Benign")
            y = np.r_[np.zeros(bm.sum()), np.ones(a.sum())]
            rowsA.append(dict(seed=seed, model=m, day=d, AUROC=roc_auc_score(y, np.r_[s[bm], s[a]])))
A = pd.DataFrame(rowsA); A.to_csv(P + "/per_day_AUROC.csv", index=False)
g = A.groupby(["model", "day"]).AUROC.agg(["mean", "std"])
g["txt"] = g["mean"].round(3).astype(str) + " ± " + g["std"].round(3).astype(str)
print("\nPER-DAY AUROC (same-day benign vs that day's attack), mean ± SD over seeds:")
print(g["txt"].unstack("day").to_string())

# ---------- (c) exact-duplicate sensitivity ----------
cols = get_num_cols(ws) + CAT
h = pd.util.hash_pandas_object(ws[cols], index=False).to_numpy()
dev = np.where((day == "2019-04-23") & (att == "Benign"))[0]
dup = np.isin(h, h[dev]) & (day >= "2019-04-24")
dd = pd.DataFrame({"day": day[day >= "2019-04-24"], "cls": att[day >= "2019-04-24"], "dup": dup[day >= "2019-04-24"]})
print("\nShare of later-day flows that exactly duplicate a development flow (by day and class):")
print(dd.groupby(["day", "cls"]).dup.mean().round(4).to_string())
rowsD = []
for seed in SEEDS:
    split = make_split(ws, seed)
    for m in M3:
        s = np.load(f"{P}/scores_{m}_seed{seed}.npy")
        tau = conformal_threshold(s[split["cal"]], 0.01)
        sets = {"FPR_" + d: (day == d) & (att == "Benign") for d in later_days}
        sets["recall_dosfam"] = np.isin(att, ["dos", "ddos"]); sets["recall_xss"] = att == "xss"
        for name, mk in sets.items():
            keep = mk & ~dup
            rowsD.append(dict(model=m, metric=name, all_flows=(s[mk] > tau).mean(),
                              no_duplicates=(s[keep] > tau).mean() if keep.sum() else np.nan))
D = pd.DataFrame(rowsD).groupby(["metric", "model"]).mean().round(4)
D.to_csv(P + "/duplicate_sensitivity.csv")
print("\nAlpha = 1%: all flows vs exact-duplicates removed (mean over seeds):")
print(D.to_string())

PAIRED BOOTSTRAP 95% CIs, seed 13, alpha = 1%:
quantity             IsolationForest             MLP minus IF            MLP minus Raw             MLP_OCSVM              RawOCSVM
target                                                                                                                            
FPR_2019-04-24  0.032 [0.032, 0.033]     0.017 [0.016, 0.018]    0.000 [-0.000, 0.000]  0.049 [0.048, 0.050]  0.049 [0.048, 0.050]
FPR_2019-04-25  0.035 [0.035, 0.036]     0.404 [0.402, 0.406]  -0.121 [-0.122, -0.119]  0.440 [0.438, 0.442]  0.560 [0.558, 0.562]
FPR_2019-04-26  0.041 [0.040, 0.042]     0.320 [0.318, 0.322]  -0.152 [-0.153, -0.151]  0.361 [0.359, 0.363]  0.513 [0.511, 0.515]
FPR_2019-04-27  0.503 [0.502, 0.505]     0.092 [0.091, 0.093]  -0.153 [-0.154, -0.151]  0.595 [0.594, 0.597]  0.748 [0.746, 0.750]
FPR_2019-04-28  0.456 [0.454, 0.458]     0.254 [0.251, 0.256]  -0.008 [-0.010, -0.007]  0.710 [0.708, 0.711]  0.718 [0.716, 0.720]
FPR_2019-04-29  0.011 [0.010, 0.011]

In [15]:
import numpy as np, pandas as pd, sys, warnings, glob, os, shutil, platform
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
P = "/content/drive/MyDrive/nfton"
OUT = P + "/paper_assets"; os.makedirs(OUT, exist_ok=True)
SEEDS = [13, 23, 37, 51, 73]
M3 = ["IsolationForest", "MLP_OCSVM", "RawOCSVM"]
ALL5 = M3 + ["MLP_OCSVM_mask0", "LightGBM"]
NAMES = {"IsolationForest": "Isolation Forest", "MLP_OCSVM": "MLP-emb. + OC-SVM (proposed)",
         "RawOCSVM": "Raw OC-SVM", "MLP_OCSVM_mask0": "MLP-emb. + OC-SVM (no masking)",
         "LightGBM": "LightGBM (supervised ref.)"}
df = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(P + "/results_*_seed*.csv"))])
df = df[df.seed.isin(SEEDS)]

def agg(model, metric, group, alpha=None):
    d = df[(df.model == model) & (df.metric == metric) & (df.group == group)]
    if alpha is not None: d = d[np.isclose(d.alpha.astype(float), alpha)]
    return d.groupby("seed").value.mean()
def fmt(x): return f"{x.mean():.3f} ± {x.std():.3f}"
def esc(t): return (str(t).replace("&", "\\&").replace("_", "\\_").replace("%", "\\%").replace("±", "$\\pm$"))
def tex(dfx, name):
    cols = list(dfx.columns)
    L = ["\\begin{tabular}{l" + "c" * len(cols) + "}", "\\toprule",
         " & ".join([""] + [esc(c) for c in cols]) + " \\\\", "\\midrule"]
    for i, r in dfx.iterrows():
        L.append(" & ".join([esc(i)] + [esc(v) for v in r.values]) + " \\\\")
    L += ["\\bottomrule", "\\end{tabular}"]
    open(f"{OUT}/{name}.tex", "w").write("\n".join(L))
    print("\n=== " + name + " ==="); print(dfx.to_string())
def save(fig, name):
    fig.savefig(f"{OUT}/{name}.png", dpi=300, bbox_inches="tight")
    fig.savefig(f"{OUT}/{name}.pdf", bbox_inches="tight"); plt.close(fig)

days_g = ["val_benign_2019-04-23"] + [f"benign_2019-04-{d}" for d in range(24, 30)]
lab = ["23 Apr (same day)"] + [f"{d} Apr" for d in range(24, 30)]

# ---------------- TABLES ----------------
tex(pd.DataFrame({NAMES[m]: {"DoS+DDoS recall": fmt(agg(m, "recall", "dos+ddos", 0.01)),
                             "XSS recall": fmt(agg(m, "recall", "xss", 0.01)),
                             "AUROC (DoS+DDoS)": fmt(agg(m, "AUROC", "dos+ddos"))} for m in ALL5}).T, "table_main_recall")
tex(pd.DataFrame({NAMES[m]: {l: fmt(agg(m, "FPR", g, 0.01)) for g, l in zip(days_g, lab)} for m in ALL5}).T, "table_main_fpr")
abl = ["IsolationForest", "MLP_OCSVM", "MLP_OCSVM_mask0", "RawOCSVM"]
tex(pd.DataFrame({NAMES[m]: {"DoS+DDoS recall": fmt(agg(m, "recall", "dos+ddos", 0.01)),
                             "AUROC (DoS+DDoS)": fmt(agg(m, "AUROC", "dos+ddos")),
                             "FPR 25 Apr": fmt(agg(m, "FPR", "benign_2019-04-25", 0.01)),
                             "FPR 29 Apr": fmt(agg(m, "FPR", "benign_2019-04-29", 0.01))} for m in abl}).T, "table_ablation")

E = pd.read_csv(P + "/E3_policies_per_seed.csv")
g = E.groupby(["model", "day"]).mean(numeric_only=True)
t3 = g[["FPR_stale", "abstain_rate", "FPR_decided", "FPR_recal_eps0.0", "FPR_recal_eps0.01"]].round(3).astype(object).fillna("n/a")
t3.columns = ["Stale FPR", "Abstain rate", "FPR if decided", "Recal. FPR (eps=0)", "Recal. FPR (eps=1%)"]
t3.index = [f"{NAMES[a]} / {b[5:]}" for a, b in t3.index]
tex(t3, "table_E3_fpr")
r3 = g.loc[g.index.get_level_values("day").isin(["2019-04-24", "2019-04-25", "2019-04-26"]),
           ["recall_stale_dosfam", "recall_recal0.0_dosfam", "recall_recal0.01_dosfam"]].round(3)
r3.columns = ["Stale recall", "Recal. recall (eps=0)", "Recal. recall (eps=1%)"]
r3.index = [f"{NAMES[a]} / {b[5:]}" for a, b in r3.index]
tex(r3, "table_E3_recall")

L2 = pd.read_csv(P + "/E5_L2.csv"); L3 = pd.read_csv(P + "/E5_L3.csv")
rows = {}
for m, lm in (("IsolationForest", "L1_IsolationForest"), ("MLP_OCSVM", "L1_MLP_OCSVM")):
    for metric, al, lb in (("recall", 0.01, "DoS+DDoS recall"), ("AUROC", None, "AUROC")):
        c, l = agg(m, metric, "dos+ddos", al), agg(lm, metric, "dos+ddos", al)
        rows[f"L1 preprocessing / {NAMES[m]} / {lb}"] = {"Clean": fmt(c), "Leaky": fmt(l), "Delta": f"{l.mean() - c.mean():+.3f}"}
for m in M3:
    d = L2[L2.model == m]
    rows[f"L2 test-label threshold / {NAMES[m]} / balanced acc."] = {"Clean": fmt(d.clean_BA), "Leaky": fmt(d.leak_BA), "Delta": f"{d.leak_BA.mean() - d.clean_BA.mean():+.3f}"}
    rows[f"L2 test-label threshold / {NAMES[m]} / realized FPR"] = {"Clean": fmt(d.clean_FPR), "Leaky": fmt(d.leak_FPR), "Delta": f"{d.leak_FPR.mean() - d.clean_FPR.mean():+.3f}"}
rows["L3 held-out selection / recall"] = {"Clean": fmt(L3.clean_recall), "Leaky": fmt(L3.leak_recall), "Delta": f"{L3.leak_recall.mean() - L3.clean_recall.mean():+.3f}"}
rows["L3 held-out selection / realized FPR"] = {"Clean": fmt(L3.clean_FPR), "Leaky": fmt(L3.leak_FPR), "Delta": f"{L3.leak_FPR.mean() - L3.clean_FPR.mean():+.3f}"}
tex(pd.DataFrame(rows).T, "table_leakage")

C = pd.read_csv(P + "/bootstrap_CI_all_seeds.csv"); c = C[(C.seed == 13) & C.target.isin(["recall_dos", "recall_ddos", "recall_xss"])].copy()
c["txt"] = c.apply(lambda r: f"{r.est:.3f} [{r.lo:.3f}, {r.hi:.3f}]", axis=1)
tci = c.pivot(index="quantity", columns="target", values="txt")
tex(tci, "table_bootstrap_seed13")
A = pd.read_csv(P + "/per_day_AUROC.csv"); ga = A.groupby(["model", "day"]).AUROC.agg(["mean", "std"])
ga["t"] = ga["mean"].round(3).astype(str) + " ± " + ga["std"].round(3).astype(str)
tauc = ga["t"].unstack("day"); tauc.index = [NAMES[i] for i in tauc.index]
tauc.columns = [c_[5:] for c_ in tauc.columns]
tex(tauc, "table_perday_auroc")

# ---------------- FIGURES ----------------
fig, ax = plt.subplots(figsize=(7.5, 4.2))
for m in M3:
    mu = [agg(m, "FPR", g_, 0.01).mean() for g_ in days_g]; sd = [agg(m, "FPR", g_, 0.01).std() for g_ in days_g]
    ax.errorbar(range(7), mu, yerr=sd, marker="o", capsize=3, label=NAMES[m])
ax.axhline(0.01, ls="--", c="k", lw=1, label="nominal target (1%)")
ax.set_xticks(range(7)); ax.set_xticklabels(lab, rotation=20); ax.set_ylabel("Realized benign false-alarm rate")
ax.set_title("False-alarm control under benign drift (frozen threshold)"); ax.legend(fontsize=8); save(fig, "fig_perday_fpr")

fig, ax = plt.subplots(figsize=(5.5, 4.5))
for m in M3 + ["LightGBM"]:
    xs = [agg(m, "FPR", "benign_later_all", a).mean() for a in (0.001, 0.01, 0.05)]
    ys = [agg(m, "recall", "dos+ddos", a).mean() for a in (0.001, 0.01, 0.05)]
    ax.plot(xs, ys, marker="o", label=NAMES[m])
ax.plot([0, 1], [0, 1], "k:", lw=1, label="no discrimination (recall = FPR)")
ax.set_xlabel("Realized benign FPR (pooled later days)"); ax.set_ylabel("DoS+DDoS recall")
ax.set_xlim(0, 0.8); ax.set_ylim(0, 1); ax.legend(fontsize=7); ax.set_title("Recall must be read next to realized FPR"); save(fig, "fig_recall_vs_fpr")

fig, axs = plt.subplots(1, 3, figsize=(12, 3.6), sharey=True)
for ax, m in zip(axs, M3):
    gm = g.loc[m]; xs = [d[5:] for d in gm.index]
    for col, l in (("FPR_stale", "stale"), ("FPR_recal_eps0.0", "recalibrated (eps=0)"), ("FPR_recal_eps0.01", "recalibrated (eps=1%)")):
        ax.plot(xs, gm[col].values, marker="o", label=l)
    ax.axhline(0.01, ls="--", c="k", lw=1); ax.set_title(NAMES[m], fontsize=9); ax.tick_params(axis="x", rotation=45)
axs[0].set_ylabel("Benign FPR (target 1%)"); axs[0].legend(fontsize=7); save(fig, "fig_E3_policies")

fig, axs = plt.subplots(1, 2, figsize=(10, 3.8))
x = np.arange(3)
cl = [L2[L2.model == m].clean_BA for m in M3]; lk = [L2[L2.model == m].leak_BA for m in M3]
axs[0].bar(x - 0.2, [v.mean() for v in cl], 0.4, yerr=[v.std() for v in cl], capsize=3, label="clean (frozen 1% threshold)")
axs[0].bar(x + 0.2, [v.mean() for v in lk], 0.4, yerr=[v.std() for v in lk], capsize=3, label="L2 (test-label threshold)")
axs[0].set_xticks(x); axs[0].set_xticklabels([NAMES[m].replace(" (proposed)", "") for m in M3], fontsize=7); axs[0].set_ylabel("Balanced accuracy"); axs[0].legend(fontsize=7)
items = [("IF clean", agg("IsolationForest", "recall", "dos+ddos", 0.01)), ("IF L1", agg("L1_IsolationForest", "recall", "dos+ddos", 0.01)),
         ("MLP clean", agg("MLP_OCSVM", "recall", "dos+ddos", 0.01)), ("MLP L1", agg("L1_MLP_OCSVM", "recall", "dos+ddos", 0.01)),
         ("L3 selected", L3.set_index("seed").leak_recall)]
axs[1].bar(range(5), [v.mean() for _, v in items], yerr=[v.std() for _, v in items], capsize=3, color=["C0", "C1", "C0", "C1", "C3"])
axs[1].set_xticks(range(5)); axs[1].set_xticklabels([n for n, _ in items], fontsize=8); axs[1].set_ylabel("DoS+DDoS recall at nominal 1%")
fig.suptitle("Leakage audit (diagnostic only; never valid detector performance)", fontsize=10); save(fig, "fig_leakage")

K = pd.read_csv(P + "/E5_L3_best_of_k.csv").groupby("k").mean_best_recall.agg(["mean", "std"])
fig, ax = plt.subplots(figsize=(4.6, 3.6)); ax.errorbar(K.index, K["mean"], yerr=K["std"], marker="o", capsize=3)
ax.set_xlabel("Configurations tried"); ax.set_ylabel("Best held-out DoS+DDoS recall"); ax.set_xticks(K.index); save(fig, "fig_inflation_vs_configs")

fig, axs = plt.subplots(1, 2, figsize=(9, 3.6))
for ax, (metric, al, yl) in zip(axs, (("recall", 0.01, "DoS+DDoS recall (alpha=1%)"), ("AUROC", None, "AUROC (DoS+DDoS)"))):
    v = [agg(m, metric, "dos+ddos", al) for m in abl]
    ax.bar(range(4), [s_.mean() for s_ in v], yerr=[s_.std() for s_ in v], capsize=3)
    ax.set_xticks(range(4)); ax.set_xticklabels(["Isolation\nForest", "MLP+SVM\nmask 10%", "MLP+SVM\nmask 0%", "Raw\nOC-SVM"], fontsize=8); ax.set_ylabel(yl)
save(fig, "fig_ablation")

import sklearn, torch, lightgbm, scipy
env = (f"python {platform.python_version()}\nnumpy {np.__version__}\npandas {pd.__version__}\nscikit-learn {sklearn.__version__}\n"
       f"torch {torch.__version__}\nlightgbm {lightgbm.__version__}\nscipy {scipy.__version__}\ncpu_count {os.cpu_count()}\nplatform {platform.platform()}")
open(OUT + "/environment.txt", "w").write(env); print("\n" + env)
shutil.make_archive(P + "/paper_assets", "zip", OUT)
print("\nFiles:", sorted(os.listdir(OUT)))
from google.colab import files
files.download(P + "/paper_assets.zip")


=== table_main_recall ===
                               DoS+DDoS recall     XSS recall AUROC (DoS+DDoS)
Isolation Forest                 0.069 ± 0.004  0.535 ± 0.003    0.695 ± 0.012
MLP-emb. + OC-SVM (proposed)     0.421 ± 0.140  0.553 ± 0.026    0.587 ± 0.018
Raw OC-SVM                       0.716 ± 0.014  0.586 ± 0.028    0.639 ± 0.015
MLP-emb. + OC-SVM (no masking)   0.493 ± 0.192  0.560 ± 0.020    0.594 ± 0.026
LightGBM (supervised ref.)       0.002 ± 0.002  0.000 ± 0.000    0.471 ± 0.081

=== table_main_fpr ===
                               23 Apr (same day)         24 Apr         25 Apr         26 Apr         27 Apr         28 Apr         29 Apr
Isolation Forest                   0.012 ± 0.003  0.034 ± 0.002  0.040 ± 0.010  0.047 ± 0.010  0.505 ± 0.013  0.501 ± 0.067  0.017 ± 0.008
MLP-emb. + OC-SVM (proposed)       0.012 ± 0.002  0.049 ± 0.004  0.428 ± 0.060  0.372 ± 0.071  0.612 ± 0.040  0.773 ± 0.069  0.097 ± 0.011
Raw OC-SVM                         0.012 ± 0.002  0.050 ± 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>